In [1]:
import torch
from torch.utils.tensorboard import SummaryWriter
import pickle
from torch import nn
from pathlib import Path
import os
from models.UNET_regression import UNetRegression
from models.DA_CNN import DA_CNN
from models.CNN_EBAM import EBAM_CNN
from models.UNET_regressionSE import UNetRegressionSE
from models.simple_CNN_regression import PixelWiseRegressor
from torch.utils.data import Subset, DataLoader
from utils.datasettemporal import TemporalDataset, TestSubsetRegression
# from utils.transforms import RescaledRotationTransform, ToTensor, Compose
from torchvision.transforms import Normalize, Compose
from utils.transforms import RescaledRotationTransform, ToTensor 
from utils.config import PROJECT_ROOT, RELEVANT_CONFIG, RAW_CONFIG
from torch.optim import AdamW
import time
from utils.splitter import train_val_test_split_temp
from torch.optim.lr_scheduler import ReduceLROnPlateau

cfg = RELEVANT_CONFIG
root = Path(PROJECT_ROOT)
submode = RELEVANT_CONFIG["submode"]

def train_loop(model, train_dataloader, optimizer, loss_fn, device):
    size = len(train_dataloader)
    total_size = len(train_dataloader.dataset)
    model.train()
    total_loss = 0
    for batch, (images, labels) in enumerate(train_dataloader):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        pred = model(images)
        loss = loss_fn(pred, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        if batch % 50 == 0:
            loss, current = loss.item(), batch * batch_size + len(images)
            print(f"loss: {loss:>12f} [{current:>6d}/{total_size:>5d}]")
    total_loss /= size
    return total_loss

In [2]:
def val_loop(val_dataloader, model, loss_fn, device, scheduler):
    model.eval()
    num_batches = len(val_dataloader)
    val_loss = 0
    size = len(val_dataloader)
    with torch.no_grad():
        for image, label in val_dataloader:
            image, label = image.to(device), label.to(device)
            pred = model(image)
            val_loss += loss_fn(pred, label).item()

    val_loss /= num_batches
    print(f"Val Loss: {val_loss:>12f} \n")
    scheduler.step(val_loss)
    return val_loss

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cuda device


In [3]:
data_aug = RescaledRotationTransform()
data = TemporalDataset(transform=data_aug)

batch_size = cfg['training']["batch_size"]
epochs = cfg['training']["epochs"]

# model = PixelWiseRegressor(data[0][0].shape[0], data[0][1].shape[0])
model = EBAM_CNN()
model = model.to(device)
optimizer = AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)

scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

# loss_fn = nn.MSELoss()

# loss_fn = nn.HuberLoss()

loss_fn = nn.L1Loss()


train_idx, val_idx, test_idx = train_val_test_split_temp(data, seed=42, test_indices_path=Path(cfg['data'][submode]["test_indices"]))


small
/mnt/c/Users/samue/SynologyDrive/OceanPropInfSatImg/data/daily_alternative_small/small_daily_alternative_sample_1993-1993.nc
Loaded existing test indices from test_indices/daily_alternative_small/small_01.pt, test size: 207200


In [4]:
train_data, val_data, = Subset(data, train_idx), Subset(data, val_idx)

In [5]:
train_data, val_data, = Subset(data, train_idx), Subset(data, val_idx)

# train_data, val_data = simple_train_val_split(data)

print(f"Train dataset size: {len(train_data)}, Val dataset size: {len(val_data)}")
print(f"Train dataset shape: img: {train_data[0][0].shape}, lbl: {train_data[0][1].shape}, Val dataset shape: img: {val_data[0][0].shape}, lbl: {val_data[0][1].shape}")

train_dataloader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
val_dataloader = DataLoader(val_data, batch_size=batch_size, shuffle=True)

best_loss = 1000000000000000000

def checkpoint(model, filename):
     torch.save(model.state_dict(), filename)

Train dataset size: 911680, Val dataset size: 124320
Train dataset shape: img: torch.Size([6, 21, 21]), lbl: torch.Size([1]), Val dataset shape: img: torch.Size([6, 21, 21]), lbl: torch.Size([1])


In [6]:
start_timestamp = time.strftime('%Y%m%d_%H%M%S')
model_name = f"MODEL:{model.name()}>TRAINSTART:{start_timestamp}>DATAFILE:{(cfg['data'][submode]['output_file']).replace('/', '_')}>STRAT:{(cfg['data'][submode]['test_indices']).replace('/', '_')}>"
model_dir = cfg["training"]["model_save_dest"]
save_dir = root / model_dir / model_name
os.makedirs(save_dir, exist_ok=True)

writer= SummaryWriter(save_dir / 'tensorboard_logs')

%load_ext tensorboard
%tensorboard --logdir {save_dir / 'tensorboard_logs'}

In [ ]:

# start_timestamp = time.strftime('%Y%m%d_%H%M%S')
# model_name = f"MODEL:{model.name()}>TRAINSTART:{start_timestamp}>DATAFILE:{(cfg['data'][submode]['output_file']).replace('/', '_')}>STRAT:{(cfg['data'][submode]['test_indices']).replace('/', '_')}>"
# model_dir = cfg["training"]["model_save_dest"]
# save_dir = root / model_dir / model_name
# os.makedirs(save_dir, exist_ok=True)

# writer= SummaryWriter(save_dir / 'tensorboard_logs')

best_epoch=0
for epoch in range(0, epochs):
        print('Epoch {}:'.format(epoch + 1))
        model.train(True)
        train_loss = train_loop(model, train_dataloader, optimizer, loss_fn, device)
        # writer.add_scalar('Loss/train', train_loss, epoch)
        writer.add_scalar('Learning Rate', optimizer.param_groups[0]['lr'], epoch)
        for name, p in model.named_parameters():
                writer.add_histogram(f"weights/{name}", p, epoch)
                if p.grad is not None:
                    writer.add_histogram(f"gradients/{name}", p.grad, epoch)
        val_loss = val_loop(val_dataloader, model, loss_fn, device, scheduler)
        writer.add_scalars('Loss', {'val': val_loss, 'train': train_loss}, epoch)
        if val_loss < best_loss:
            best_epoch = epoch
            best_loss = val_loss
            corresponding_train_loss = train_loss
            model_state_path = save_dir / 'best_model_state'
            model_path = save_dir / 'best_model'
            torch.save(model.state_dict(), model_state_path)
            torch.save(model, model_path)
        if epoch - best_epoch >= cfg["training"]["early_stopping_thresh"]:
             print(f"Early stopping on epoch {epoch}")
             break
print(f"Best loss: {best_loss}")



Epoch 1:
loss:   133.925552 [    50/911680]
loss:   121.682518 [  2550/911680]
loss:    72.699493 [  5050/911680]
loss:   154.432297 [  7550/911680]
loss:   111.234734 [ 10050/911680]
loss:    67.530312 [ 12550/911680]
loss:    28.633898 [ 15050/911680]
loss:    26.976048 [ 17550/911680]
loss:    40.428223 [ 20050/911680]
loss:    12.351737 [ 22550/911680]
loss:    39.274467 [ 25050/911680]
loss:    43.077202 [ 27550/911680]
loss:    21.287123 [ 30050/911680]
loss:     7.706676 [ 32550/911680]
loss:    44.284248 [ 35050/911680]
loss:    12.608617 [ 37550/911680]
loss:    13.099074 [ 40050/911680]
loss:    22.361816 [ 42550/911680]
loss:    22.823639 [ 45050/911680]
loss:    41.421761 [ 47550/911680]
loss:    18.200682 [ 50050/911680]
loss:     7.646594 [ 52550/911680]
loss:    29.906683 [ 55050/911680]
loss:    38.263794 [ 57550/911680]
loss:    24.418457 [ 60050/911680]
loss:     9.210509 [ 62550/911680]
loss:    23.745068 [ 65050/911680]
loss:    38.832287 [ 67550/911680]
loss:    12

In [ ]:
info = {
    "start_time": start_timestamp,
    "data_file": f"{cfg['data']['data_dir']}/{cfg['data'][submode]['output_file']}",
    "test_indices": f"{cfg['data'][submode]['test_indices']}",
    "epochs": epochs,
    "batch_size": batch_size,
    "model": model.__repr__(),
    "optimizer": optimizer.__repr__(),
    "loss_fn": loss_fn.__repr__(),
    "scheduler": scheduler.__repr__(),
    "best_loss": best_loss,
    "best_epoch": best_epoch,
    "corresponding_train_loss": corresponding_train_loss,
    "train_dataset_size": len(train_data),
    "val_dataset_size": len(val_data),
    "test_dataset_size": len(test_idx),
    "transform": data.transform.__repr__() if data.transform else None,
    "target_transform": data.target_transform.__repr__() if data.target_transform else None,
    "downsample": data.downsample if hasattr(data, 'downsample') else None,
    "grid_size": data.grid_size if hasattr(data, 'grid_size') else None
}

In [ ]:
info_path =  save_dir / 'training_info.txt'
with open(info_path, 'w') as f:
    for key, value in info.items():
        f.write(f"{key}: {value}\n")
print(f"Training completed. Best model saved at {model_path}")

Training completed. Best model saved at /mnt/c/Users/samue/SynologyDrive/OceanPropInfSatImg/saved_models/saved_daily_alternative_small_models/MODEL:PixelWiseRegressor>TRAINSTART:20250604_181302>DATAFILE:small_daily_alternative_sample_1993-1993.nc>STRAT:test_indices_daily_alternative_small_small_01.pt>/best_model


In [ ]:
if cfg["training"]["immediate_test"]:
    test_data = TestSubsetRegression(data, test_idx)
    test_dataloader = torch.utils.data.DataLoader(
    test_data,
    batch_size=1,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)
    model.eval()
    loss = 0

    after_model = []

    filepath = save_dir/"results.pkl"

    with torch.no_grad():
        for i, batch in enumerate(test_dataloader):
            images, labels, metadata = batch
            images_gpu = images.to(device)
            preds = model(images_gpu)
            loss += loss_fn(preds, labels.to(device)).item()
            preds, labels = preds.cpu(), labels.cpu()
            batch_dict = {"image": images, "label": labels, "pred": preds, "grid": metadata[0], "centre": metadata[1], "month": metadata[2]}
            after_model.append(batch_dict)

            if i % 100 == 0:
                print(f"Processed {i} batches")

            # Periodically append new results
            if i % 10000 == 0 and i > 0:
                with open(filepath, "ab") as f:
                    pickle.dump(after_model, f)
                print(f"Appended {i} batches")
                after_model.clear()

        # if i > num_samples:
        #     break
    # Save any remaining batches
        if len(after_model) > 0:
            with open(filepath, "ab") as f:
                pickle.dump(after_model, f)
print(f"Total loss: {loss / len(test_dataloader)}")
with open(info_path, 'a') as f:
    f.write(f"total_test_loss: {loss / len(test_dataloader)}\n")


Processed 0 batches
Processed 100 batches
Processed 200 batches
Processed 300 batches
Processed 400 batches
Processed 500 batches
Processed 600 batches
Processed 700 batches
Processed 800 batches
Processed 900 batches
Processed 1000 batches
Processed 1100 batches
Processed 1200 batches
Processed 1300 batches
Processed 1400 batches
Processed 1500 batches
Processed 1600 batches
Processed 1700 batches
Processed 1800 batches
Processed 1900 batches
Processed 2000 batches
Processed 2100 batches
Processed 2200 batches
Processed 2300 batches
Processed 2400 batches
Processed 2500 batches
Processed 2600 batches
Processed 2700 batches
Processed 2800 batches
Processed 2900 batches
Processed 3000 batches
Processed 3100 batches
Processed 3200 batches
Processed 3300 batches
Processed 3400 batches
Processed 3500 batches
Processed 3600 batches
Processed 3700 batches
Processed 3800 batches
Processed 3900 batches
Processed 4000 batches
Processed 4100 batches
Processed 4200 batches
Processed 4300 batches
